# 🔬 TrustKarat — Audio Engineering Pipeline

> **Tap-test purity detection**: 14K / 22K / 24K gold classification  
> **Method**: MFCC + Extended Spectral Features + SVM (RBF kernel)  
> **Published baseline accuracy**: 94.58%

---

### Pipeline Overview

| Stage | Description |
|-------|-------------|
| **1. Feature Extraction** | MFCC + Δ + ΔΔ, spectral descriptors, chroma, rhythm, ZCR, RMS decay, resonance |
| **2. Dataset Loading** | Class-balanced loader with on-the-fly augmentation (noise, pitch, stretch, volume) |
| **3. Model Training** | GridSearchCV over SVM · comparison vs RF + GBT · 5-fold cross-validation |
| **4. Inference** | Single-file prediction with full confidence breakdown |
| **5. Waveform Analysis** | Per-class waveform · Mel-spectrogram · RMS decay visualisation |

### Expected Directory Layout

```
dataset/
├── 14K/   *.wav / .mp3 / .ogg / .flac
├── 22K/   ...
└── 24K/   ...
```

---
## 0. Setup & Imports

In [1]:
# ── Install dependencies (uncomment if needed) ────────────────────────────────
# !pip install librosa scikit-learn seaborn joblib pandas matplotlib

import os
import warnings
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import librosa
import librosa.display
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm                 import SVC
from sklearn.ensemble            import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing       import StandardScaler, LabelEncoder
from sklearn.model_selection     import (StratifiedKFold, cross_val_score,
                                          GridSearchCV, train_test_split)
from sklearn.metrics             import (classification_report, confusion_matrix,
                                          accuracy_score)
from sklearn.pipeline            import Pipeline

warnings.filterwarnings("ignore")
print("✅ All imports successful.")

✅ All imports successful.


---
## 1. Global Configuration

Centralised constants — change once, applies everywhere.

In [2]:
# ── Audio processing ──────────────────────────────────────────────────────────
SR          = 22_050      # sample rate (Hz)
DURATION    = 3.0         # seconds to analyse per clip
N_MFCC      = 40          # MFCC coefficients
HOP_LENGTH  = 512
N_FFT       = 2048
N_MELS      = 128

# ── Classes & paths ───────────────────────────────────────────────────────────
CLASSES     = ["14K", "22K", "24K"]
AUDIO_EXTS  = {".wav", ".mp3", ".ogg", ".flac", ".m4a", ".aac"}

MODEL_PATH  = "models/svm_purity.pkl"
SCALER_PATH = "models/scaler.pkl"
REPORT_DIR  = "reports"

# ── Palette (dark gold theme) ─────────────────────────────────────────────────
GOLD     = "#C9A84C"
BG_DARK  = "#09110E"
BG_MID   = "#0F1C14"
BORDER   = "#1e3020"
MUTED    = "#6B8070"
CLASS_COLORS = ["#3A80D2", "#C9A84C", "#2D9E5F"]

os.makedirs("models",   exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print(f"Sample rate : {SR:,} Hz")
print(f"Clip length : {DURATION}s  →  {int(SR*DURATION):,} samples")
print(f"MFCC dims   : {N_MFCC}  (×3 with Δ/ΔΔ = {N_MFCC*3} × 2 stats = {N_MFCC*6})")
print(f"Classes     : {CLASSES}")

Sample rate : 22,050 Hz
Clip length : 3.0s  →  66,150 samples
MFCC dims   : 40  (×3 with Δ/ΔΔ = 120 × 2 stats = 240)
Classes     : ['14K', '22K', '24K']


---
## 2. Feature Extraction

Each clip yields a **~292-dimensional** feature vector across seven groups:

| Group | Features | Dimensions | Physical meaning |
|-------|----------|-----------|------------------|
| A — MFCC + Δ + ΔΔ | mean & std per coefficient | 240 | Timbral texture |
| B — Spectral descriptors | centroid, bandwidth, roll-off, flatness, contrast | 15 | Frequency shape |
| C — Chroma STFT | mean & std per pitch class | 24 | Harmonic content |
| D — Rhythm | tempo + onset strength | 2 | Temporal pattern |
| E — Zero crossing rate | mean + std | 2 | Tap sharpness |
| F — RMS energy | mean, max, std, slope | 4 | Amplitude decay |
| G — Resonance decay | T10 / T20 / T50 | 3 | Ring-off time 🔑 |

> **Key insight:** Higher-karat gold sustains resonance longer (slower RMS slope, higher T50).

In [3]:
def extract_features(file_path: str, augment: bool = False) -> np.ndarray | None:
    """
    Extract a rich feature vector from one audio file.

    Parameters
    ----------
    file_path : str   Path to audio file.
    augment   : bool  Apply random augmentation (training only).

    Returns
    -------
    np.ndarray of shape (292,), dtype float32  —  or None on error.
    """
    try:
        y, sr = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)

        # Pad to fixed length
        target_len = int(SR * DURATION)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))

        if augment:
            y = _augment(y, sr)

        feats = []

        # ── A. MFCC + Delta + Delta-Delta ───────────────────────────────
        mfcc    = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                        n_fft=N_FFT, hop_length=HOP_LENGTH)
        d_mfcc  = librosa.feature.delta(mfcc)
        dd_mfcc = librosa.feature.delta(mfcc, order=2)

        for m in [mfcc, d_mfcc, dd_mfcc]:
            feats.extend(np.mean(m, axis=1))  # 40 dims
            feats.extend(np.std(m,  axis=1))  # 40 dims → 240 total

        # ── B. Spectral descriptors ─────────────────────────────────────
        spec_centroid  = librosa.feature.spectral_centroid(
                            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        spec_bandwidth = librosa.feature.spectral_bandwidth(
                            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        spec_rolloff   = librosa.feature.spectral_rolloff(
                            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        spec_flatness  = librosa.feature.spectral_flatness(
                            y=y,        n_fft=N_FFT, hop_length=HOP_LENGTH)
        spec_contrast  = librosa.feature.spectral_contrast(
                            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)

        for feat in [spec_centroid, spec_bandwidth, spec_rolloff, spec_flatness]:
            feats.append(float(np.mean(feat)))
            feats.append(float(np.std(feat)))

        feats.extend(np.mean(spec_contrast, axis=1))  # 7 sub-bands

        # ── C. Chroma STFT ──────────────────────────────────────────────
        chroma = librosa.feature.chroma_stft(
                    y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        feats.extend(np.mean(chroma, axis=1))  # 12 dims
        feats.extend(np.std(chroma,  axis=1))  # 12 dims

        # ── D. Rhythm ───────────────────────────────────────────────────
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
        feats.append(float(tempo))
        onset_strength = librosa.onset.onset_strength(y=y, sr=sr)
        feats.append(float(np.mean(onset_strength)))

        # ── E. Zero Crossing Rate ───────────────────────────────────────
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
        feats.append(float(np.mean(zcr)))
        feats.append(float(np.std(zcr)))

        # ── F. RMS Energy & Decay Envelope ─────────────────────────────
        rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)[0]
        feats.append(float(np.mean(rms)))
        feats.append(float(np.max(rms)))
        feats.append(float(np.std(rms)))
        # Decay slope — KEY: higher karat gold decays slower (more negative slope = faster)
        slope = float(np.polyfit(np.arange(len(rms)), rms, 1)[0]) if len(rms) > 1 else 0.0
        feats.append(slope)

        # ── G. Resonance Decay — ring-off time ─────────────────────────
        peak_idx = int(np.argmax(np.abs(y)))
        peak_amp = float(np.max(np.abs(y)))
        after    = np.abs(y[peak_idx:])

        def decay_time(ratio: float) -> float:
            """Seconds until amplitude drops below ratio × peak."""
            idxs = np.where(after < peak_amp * ratio)[0]
            return float(idxs[0] / SR) if len(idxs) > 0 else DURATION

        feats.append(decay_time(0.10))  # T90 — 90 % decay
        feats.append(decay_time(0.20))  # T80 — 80 % decay
        feats.append(decay_time(0.50))  # T50 — 50 % decay (half-life)

        return np.array(feats, dtype=np.float32)

    except Exception as exc:
        print(f"  ⚠  Feature error [{file_path}]: {exc}")
        return None


print(f"✅ extract_features defined — expected output: ~292 dims per file")

✅ extract_features defined — expected output: ~292 dims per file


### 2a. Data Augmentation

Four transforms applied **randomly at training time** to increase diversity without collecting new samples:

| Transform | Effect | Rationale |
|-----------|--------|-----------|
| White noise | Simulates different tapping surfaces | Surface texture variance |
| Pitch shift ±1 semitone | Small frequency variation | Jewellery shape/size variance |
| Time stretch ±10 % | Speed variation | Tap force variance |
| Volume shift ×0.8–1.2 | Amplitude variation | Microphone distance variance |

In [4]:
def _augment(y: np.ndarray, sr: int) -> np.ndarray:
    """Apply one randomly chosen augmentation."""
    target = int(SR * DURATION)
    choice = np.random.randint(4)

    if choice == 0:   # White noise
        y = y + np.random.randn(len(y)) * 0.003

    elif choice == 1: # Pitch shift ±1 semitone
        steps = np.random.choice([-1, 1])
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

    elif choice == 2: # Time stretch ±10 %
        rate = np.random.uniform(0.9, 1.1)
        y    = librosa.effects.time_stretch(y, rate=rate)
        y    = y[:target] if len(y) >= target else np.pad(y, (0, target - len(y)))

    elif choice == 3: # Volume shift
        y = y * np.random.uniform(0.8, 1.2)

    return y

print("✅ _augment defined — 4 transforms available")

✅ _augment defined — 4 transforms available


---
## 3. Dataset Loading

In [18]:

import imageio_ffmpeg
import subprocess
import time
from pathlib import Path
from collections import Counter

def convert_m4a_to_wav(root_dir: str):
    """
    Convert all .m4a files under root_dir to .wav using the bundled ffmpeg.
    Skips files that have already been converted.
    """
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    files  = list(Path(root_dir).rglob("*.m4a"))

    if not files:
        print("ℹ️  No .m4a files found — skipping conversion.")
        return

    print(f"🔄 Converting {len(files)} .m4a files to .wav...")
    ok, fail = 0, 0

    for i, f in enumerate(files, 1):
        out = f.with_suffix(".wav")
        if out.exists():
            ok += 1
            continue
        result = subprocess.run(
            [ffmpeg, "-y", "-i", str(f), str(out)],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            ok += 1
        else:
            fail += 1
            print(f"  ⚠  {f.name}: {result.stderr[-200:]}")

        if i % 50 == 0:
            print(f"  {i}/{len(files)} processed...")

    print(f"✅ Conversion done — {ok} succeeded, {fail} failed")


AUDIO_EXTS = {".wav"}   # .m4a converted — load .wav only

def load_dataset(
    data_dir  : str,
    augment   : bool = True,
    aug_factor: int  = 2,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load audio files from <data_dir>/14K/, /22K/, /24K/.

    Parameters
    ----------
    data_dir   : Root directory containing class subfolders.
    augment    : Whether to generate augmented copies.
    aug_factor : Number of augmented copies per original file.

    Returns
    -------
    X : np.ndarray  shape (n_samples, n_features)
    y : np.ndarray  shape (n_samples,)  — string labels
    """
    data_dir = Path(data_dir)
    X, y = [], []

    for class_name in CLASSES:
        class_dir = data_dir / class_name
        if not class_dir.exists():
            print(f"  ⚠  Missing: {class_dir}")
            continue

        files = [f for f in class_dir.iterdir() if f.suffix.lower() in AUDIO_EXTS]
        print(f"  {class_name}: {len(files)} .wav files found")

        for fpath in files:
            feat = extract_features(str(fpath), augment=False)
            if feat is not None:
                X.append(feat)
                y.append(class_name)

            if augment:
                for _ in range(aug_factor):
                    feat_aug = extract_features(str(fpath), augment=True)
                    if feat_aug is not None:
                        X.append(feat_aug)
                        y.append(class_name)

    X_arr = np.array(X)
    y_arr = np.array(y)

    print(f"\n  ✅ Feature matrix     : {X_arr.shape}")
    print(f"  Class distribution  : {dict(Counter(y_arr))}")
    return X_arr, y_arr


# ── Step 3: Run ───────────────────────────────────────────────────────────────
DATA_DIR = r"E:\TrustKarat\SoundDataset\train"   # ← your dataset path

convert_m4a_to_wav(DATA_DIR)

print("\n📂 Loading dataset...")
t0 = time.time()
X_raw, y_raw = load_dataset(DATA_DIR, augment=True, aug_factor=2)
print(f"⏱  Done in {time.time()-t0:.1f}s")

🔄 Converting 720 .m4a files to .wav...
  50/720 processed...
  100/720 processed...
  150/720 processed...
  200/720 processed...
✅ Conversion done — 720 succeeded, 0 failed

📂 Loading dataset...
  14K: 240 .wav files found
  22K: 240 .wav files found
  24K: 240 .wav files found

  ✅ Feature matrix     : (2160, 290)
  Class distribution  : {np.str_('14K'): 720, np.str_('22K'): 720, np.str_('24K'): 720}
⏱  Done in 153.7s


---
## 4. Preprocessing

In [19]:
# ── Encode labels ─────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(y_raw)
joblib.dump(le, "models/label_encoder.pkl")
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ── Scale features ────────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
joblib.dump(scaler, SCALER_PATH)
print(f"Scaled X: mean ≈ {X_scaled.mean():.4f}, std ≈ {X_scaled.std():.4f}")

# ── Train / Test split ────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train : {X_tr.shape[0]} samples")
print(f"Test  : {X_te.shape[0]} samples")

Label mapping: {np.str_('14K'): np.int64(0), np.str_('22K'): np.int64(1), np.str_('24K'): np.int64(2)}
Scaled X: mean ≈ -0.0000, std ≈ 1.0000
Train : 1728 samples
Test  : 432 samples


---
## 5. Hyperparameter Search (SVM)

5-fold stratified grid search over:

| Parameter | Values |
|-----------|--------|
| `C` | 0.1 · 1 · 10 · 100 |
| `gamma` | scale · auto · 0.001 · 0.01 |
| `kernel` | RBF |

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "C"     : [0.1, 1, 10, 100],
    "gamma" : ["scale", "auto", 0.001, 0.01],
    "kernel": ["rbf"],
}

grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv      = cv,
    scoring = "accuracy",
    n_jobs  = -1,
    verbose = 1,
)

print("🔍 Running grid search...")
grid.fit(X_scaled, y)

best_svm = grid.best_estimator_
print(f"\n  Best params   : {grid.best_params_}")
print(f"  Best CV acc   : {grid.best_score_:.4f}")

🔍 Running grid search...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

  Best params   : {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
  Best CV acc   : 0.9889


### 5a. CV Result Heatmap
Visualise the full grid to see how C and gamma interact.

In [21]:
cv_results = pd.DataFrame(grid.cv_results_)
pivot = cv_results.pivot_table(
    index   = "param_C",
    columns = "param_gamma",
    values  = "mean_test_score",
)

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_MID)
sns.heatmap(
    pivot, annot=True, fmt=".3f", cmap="YlOrBr",
    ax=ax, linewidths=0.5, linecolor=BORDER,
    cbar_kws={"shrink": 0.8}
)
ax.set_title("GridSearch CV Accuracy — C vs gamma", color=GOLD, pad=10, fontsize=12)
ax.set_xlabel("gamma", color=MUTED)
ax.set_ylabel("C",     color=MUTED)
ax.tick_params(colors=MUTED)
plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/gridsearch_heatmap.png", dpi=150,
            bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"Saved → {REPORT_DIR}/gridsearch_heatmap.png")

Saved → reports/gridsearch_heatmap.png


---
## 6. Classifier Comparison

5-fold CV across SVM (tuned), Random Forest, and Gradient Boosting.

In [22]:
models = {
    "SVM (tuned)"      : best_svm,
    "Random Forest"    : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, random_state=42),
}

cv_results_compare = {}
print(f"{'Model':25s}  Mean ± Std")
print("-" * 42)
for name, clf in models.items():
    scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_results_compare[name] = scores
    print(f"  {name:23s}: {scores.mean():.4f} ± {scores.std():.4f}")

Model                      Mean ± Std
------------------------------------------
  SVM (tuned)            : 0.9889 ± 0.0042
  Random Forest          : 0.9736 ± 0.0093
  Gradient Boosting      : 0.9801 ± 0.0082


In [25]:
# ── Classifier Comparison Plot ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_MID)

for i, (name, scores) in enumerate(cv_results_compare.items()):
    bar = ax.bar(
        name, scores.mean(), color=CLASS_COLORS[i], alpha=0.85,
        yerr=scores.std(), capsize=5, error_kw={"ecolor": MUTED}
    )
    ax.text(i, scores.mean() + 0.005, f"{scores.mean():.3f}",
            ha="center", color=CLASS_COLORS[i], fontsize=10, fontweight="bold")

ax.set_ylim(0.7, 1.0)
ax.set_ylabel("5-Fold CV Accuracy", color=MUTED)
ax.set_title("Classifier Comparison", color=GOLD, pad=10, fontsize=12)
ax.tick_params(colors=MUTED)
for spine in ["bottom", "left"]:  ax.spines[spine].set_color(BORDER)
for spine in ["top",    "right"]: ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/classifier_comparison.png", dpi=150,
            bbox_inches="tight", facecolor=BG_DARK)
plt.show()

---
## 7. Final Training & Evaluation

In [26]:
# ── Train final model on full scaled dataset ──────────────────────────────────
print("💾 Training final SVM on full dataset...")
best_svm.fit(X_scaled, y)
joblib.dump(best_svm, MODEL_PATH)
print(f"  Saved → {MODEL_PATH}")

# ── Evaluate on held-out test set ────────────────────────────────────────────
best_svm.fit(X_tr, y_tr)        # Re-fit on train split for clean test eval
y_pred = best_svm.predict(X_te)

acc = accuracy_score(y_te, y_pred)
print(f"\n  Test accuracy : {acc:.4f}")
print()
print(classification_report(y_te, y_pred, target_names=le.classes_))

💾 Training final SVM on full dataset...
  Saved → models/svm_purity.pkl

  Test accuracy : 0.9884

              precision    recall  f1-score   support

         14K       1.00      0.99      1.00       144
         22K       0.99      0.97      0.98       144
         24K       0.97      1.00      0.99       144

    accuracy                           0.99       432
   macro avg       0.99      0.99      0.99       432
weighted avg       0.99      0.99      0.99       432



### 7a. Confusion Matrix

In [27]:
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_MID)

sns.heatmap(
    cm, annot=True, fmt="d", cmap="YlOrBr",
    xticklabels=le.classes_, yticklabels=le.classes_,
    ax=ax, linewidths=0.5, linecolor=BORDER,
    cbar_kws={"shrink": 0.8}
)
ax.set_xlabel("Predicted", color=GOLD)
ax.set_ylabel("Actual",    color=GOLD)
ax.set_title("Confusion Matrix", color=GOLD, pad=10, fontsize=12)
ax.tick_params(colors=MUTED)

plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/confusion_matrix.png", dpi=150,
            bbox_inches="tight", facecolor=BG_DARK)
plt.show()

### 7b. Feature Importance (via Random Forest proxy)

SVM with an RBF kernel does not produce native feature importances. We train a Random Forest on the same data as a proxy — the importances are broadly representative.

In [28]:
rf_proxy = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_proxy.fit(X_scaled, y)
imps = rf_proxy.feature_importances_

# Group importances by semantic feature block
BLOCKS = {
    "MFCC mean"       : (0,   40),
    "MFCC std"        : (40,  80),
    "Delta mean"      : (80,  120),
    "Delta std"       : (120, 160),
    "Delta² mean"     : (160, 200),
    "Delta² std"      : (200, 240),
    "Spectral"        : (240, 255),
    "Chroma"          : (255, 279),
    "Rhythm / ZCR / RMS": (279, 289),
    "Resonance decay" : (289, 292),
}

block_imps = {
    k: float(np.sum(imps[v[0]:v[1]]))
    for k, v in BLOCKS.items()
}
sorted_blocks = sorted(block_imps.items(), key=lambda x: x[1], reverse=True)

labels = [b[0] for b in sorted_blocks]
vals   = [b[1] for b in sorted_blocks]
colors_bar = [GOLD if ("Resonance" in l or "MFCC" in l) else "#2D9E5F" for l in labels]

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_MID)
ax.barh(labels, vals, color=colors_bar, alpha=0.85)
ax.set_xlabel("Aggregate Feature Importance", color=MUTED)
ax.set_title("What Matters Most for Purity Detection", color=GOLD, pad=10, fontsize=12)
ax.tick_params(colors=MUTED)
for sp in ax.spines.values(): sp.set_color(BORDER)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/feature_importance.png", dpi=150,
            bbox_inches="tight", facecolor=BG_DARK)
plt.show()

# Print table
print(f"{'Feature Block':25s}  Importance")
print("-" * 40)
for k, v in sorted_blocks:
    print(f"  {k:23s}: {v:.4f}")

Feature Block              Importance
----------------------------------------
  MFCC mean              : 0.2711
  Rhythm / ZCR / RMS     : 0.1880
  MFCC std               : 0.1673
  Chroma                 : 0.0889
  Delta std              : 0.0860
  Delta² std             : 0.0704
  Delta mean             : 0.0535
  Spectral               : 0.0430
  Delta² mean            : 0.0307
  Resonance decay        : 0.0011


---
## 8. Waveform Pattern Analysis

Visual comparison of per-class:
- **Row 1** — Average waveform
- **Row 2** — Mel spectrogram
- **Row 3** — RMS decay curves (individual + mean)

In [29]:
def analyse_waveform_patterns(data_dir: str, max_files_per_class: int = 10):
    """
    Generate a 3×3 figure comparing waveform characteristics per karat class.

    Parameters
    ----------
    data_dir              : Root directory with class subfolders.
    max_files_per_class   : Cap files sampled for speed.
    """
    data_dir = Path(data_dir)
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    fig.patch.set_facecolor(BG_DARK)

    for col, class_name in enumerate(CLASSES):
        class_dir = data_dir / class_name
        if not class_dir.exists():
            print(f"  ⚠  Missing: {class_dir}")
            continue

        files = [f for f in class_dir.iterdir()
                 if f.suffix.lower() in AUDIO_EXTS][:max_files_per_class]

        waves, decays = [], []
        for fpath in files:
            try:
                y, sr = librosa.load(str(fpath), sr=SR, duration=DURATION)
                target = int(SR * DURATION)
                if len(y) < target:
                    y = np.pad(y, (0, target - len(y)))
                waves.append(y)
                rms = librosa.feature.rms(y=y)[0]
                if len(rms) > 1:
                    decays.append(float(np.polyfit(np.arange(len(rms)), rms, 1)[0]))
            except Exception:
                continue

        if not waves:
            continue

        avg_wave = np.mean(waves, axis=0)
        t        = np.linspace(0, DURATION, len(avg_wave))
        col_c    = CLASS_COLORS[col]

        # ── Row 0: Waveform ───────────────────────────────────────────
        ax = axes[0][col]
        ax.set_facecolor(BG_MID)
        ax.plot(t, avg_wave, color=col_c, linewidth=0.8, alpha=0.9)
        ax.fill_between(t, avg_wave, alpha=0.15, color=col_c)
        ax.set_title(f"{class_name} — Waveform", color=GOLD, fontsize=11, pad=6)
        ax.tick_params(colors=MUTED)
        for sp in ax.spines.values(): sp.set_color(BORDER)

        # ── Row 1: Mel Spectrogram ────────────────────────────────────
        ax = axes[1][col]
        ax.set_facecolor(BG_MID)
        mel    = librosa.feature.melspectrogram(y=avg_wave, sr=SR, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(
            mel_db, sr=SR, hop_length=HOP_LENGTH,
            x_axis="time", y_axis="mel", ax=ax, cmap="magma"
        )
        ax.set_title(f"{class_name} — Mel Spectrogram", color=GOLD, fontsize=11, pad=6)
        ax.tick_params(colors=MUTED)
        for sp in ax.spines.values(): sp.set_color(BORDER)

        # ── Row 2: RMS Decay Curves ───────────────────────────────────
        ax = axes[2][col]
        ax.set_facecolor(BG_MID)
        rms_list = []
        for w in waves[:5]:
            rms   = librosa.feature.rms(y=w, hop_length=HOP_LENGTH)[0]
            t_rms = np.linspace(0, DURATION, len(rms))
            ax.plot(t_rms, rms, color=col_c, alpha=0.35, linewidth=1)
            rms_list.append(rms)
        if rms_list:
            min_len  = min(len(r) for r in rms_list)
            avg_rms  = np.mean([r[:min_len] for r in rms_list], axis=0)
            t_rms    = np.linspace(0, DURATION, min_len)
            ax.plot(t_rms, avg_rms, color=col_c, linewidth=2.0, label="Mean")
        ax.set_title(f"{class_name} — RMS Decay", color=GOLD, fontsize=11, pad=6)
        avg_decay = np.mean(decays) if decays else 0
        ax.text(0.6, 0.85, f"slope: {avg_decay:.4f}",
                transform=ax.transAxes, color=MUTED, fontsize=8)
        ax.tick_params(colors=MUTED)
        for sp in ax.spines.values(): sp.set_color(BORDER)

    plt.suptitle("TrustKarat — Waveform Pattern Analysis by Karat Class",
                 color=GOLD, fontsize=13, y=1.01)
    plt.tight_layout()
    out = f"{REPORT_DIR}/waveform_patterns.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG_DARK)
    plt.show()
    print(f"Saved → {out}")


print("🎨 Generating waveform pattern analysis...")
analyse_waveform_patterns(DATA_DIR)

🎨 Generating waveform pattern analysis...
Saved → reports/waveform_patterns.png


---
## 9. Inference — Single File Prediction

Loads the trained model and returns a structured prediction dict ready for the TrustKarat API.

In [30]:
PURITY_MAP = {"14K": 58.5, "22K": 91.6, "24K": 99.9}
RISK_MAP   = {"14K": "Medium", "22K": "Low", "24K": "Low"}


def predict(file_path: str) -> dict:
    """
    Predict gold purity class from a single audio file.

    Returns
    -------
    dict with keys:
        purity_class   : "14K" | "22K" | "24K"
        purity_percent : float  (e.g. 99.9 for 24K)
        confidence     : float  (0–1)
        risk           : "Low" | "Medium"
        method         : str    (model identifier)
        probabilities  : dict   per-class softmax probabilities
    """
    for path in [MODEL_PATH, SCALER_PATH, "models/label_encoder.pkl"]:
        if not os.path.exists(path):
            return {"error": f"Missing artefact: {path}. Run training first."}

    clf    = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    le     = joblib.load("models/label_encoder.pkl")

    feats = extract_features(file_path, augment=False)
    if feats is None:
        return {"error": "Feature extraction failed"}

    X_in  = scaler.transform(feats.reshape(1, -1))
    pred  = clf.predict(X_in)[0]
    proba = clf.predict_proba(X_in)[0]
    label = le.inverse_transform([pred])[0]

    return {
        "purity_class"  : label,
        "purity_percent": PURITY_MAP.get(label, 0),
        "confidence"    : round(float(np.max(proba)), 3),
        "risk"          : RISK_MAP.get(label, "Medium"),
        "method"        : "svm_mfcc_extended",
        "probabilities" : {
            cls: round(float(p), 3)
            for cls, p in zip(le.classes_, proba)
        },
    }


print("✅ predict() defined")

# ── Example usage ─────────────────────────────────────────────────────────────
# result = predict("path/to/tap.wav")
# for k, v in result.items():
#     print(f"  {k:20s}: {v}")

✅ predict() defined


### 9a. Confidence visualisation for a single prediction

In [31]:
def plot_prediction(result: dict):
    """Render a confidence bar chart for a predict() result dict."""
    if "error" in result:
        print(f"❌ {result['error']}")
        return

    probs  = result["probabilities"]
    labels = list(probs.keys())
    values = list(probs.values())
    colors = [GOLD if l == result["purity_class"] else MUTED for l in labels]

    fig, ax = plt.subplots(figsize=(5, 3))
    fig.patch.set_facecolor(BG_DARK)
    ax.set_facecolor(BG_MID)
    ax.barh(labels, values, color=colors, alpha=0.9)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Probability", color=MUTED)
    ax.set_title(
        f"Prediction: {result['purity_class']}  "
        f"({result['purity_percent']}% gold)  —  "
        f"conf {result['confidence']:.1%}",
        color=GOLD, fontsize=11, pad=8
    )
    ax.tick_params(colors=MUTED)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    plt.tight_layout()
    plt.show()


# ── Example ───────────────────────────────────────────────────────────────────
# result = predict("path/to/tap.wav")
# plot_prediction(result)
print("✅ plot_prediction() defined — call with the output of predict()")

✅ plot_prediction() defined — call with the output of predict()


---
## 10. Summary

Collect all key metrics into one table.

In [32]:
summary = {
    "Dataset size (total samples)" : len(X_raw),
    "Feature dimensions"           : X_raw.shape[1],
    "Best SVM params"              : str(grid.best_params_),
    "Best CV accuracy (SVM)"       : f"{grid.best_score_:.4f}",
    "Test accuracy"                : f"{acc:.4f}",
    "Model saved at"               : MODEL_PATH,
    "Scaler saved at"              : SCALER_PATH,
    "Reports directory"            : REPORT_DIR,
}

print("\n" + "═"*55)
print("  TrustKarat Pipeline — Run Summary")
print("═"*55)
for k, v in summary.items():
    print(f"  {k:35s}: {v}")
print("═"*55)


═══════════════════════════════════════════════════════
  TrustKarat Pipeline — Run Summary
═══════════════════════════════════════════════════════
  Dataset size (total samples)       : 2160
  Feature dimensions                 : 290
  Best SVM params                    : {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
  Best CV accuracy (SVM)             : 0.9889
  Test accuracy                      : 0.9884
  Model saved at                     : models/svm_purity.pkl
  Scaler saved at                    : models/scaler.pkl
  Reports directory                  : reports
═══════════════════════════════════════════════════════
